# Module 3, Section 3: CI/CD Regression Gating

In Module 2 we built a real eval suite: a curated dataset (`baseline_dataset.json`), two evaluators (`correctness_evaluator` and `count_total_tool_calls_evaluator`), and a workflow for eval-driven development — measure baseline, identify a weakness, improve the agent, re-measure.

But all of that ran manually, in a notebook, whenever we remembered to run it. Nothing stops a teammate from merging a PR that quietly breaks the agent's correctness. That's exactly the gap **CI regression gating** closes: the same eval suite from Module 2, wired into GitHub Actions so it runs automatically on every PR and **fails the check** if the agent's correctness drops below a threshold — the same way a unit test failure would.

In this section we'll walk through:
1. Why a single, persistent LangSmith dataset (not a throwaway one per run) matters for comparing experiments over time
2. `evals/run_ci_eval.py` — the script that actually runs in CI
3. `.github/workflows/eval-regression.yml` — the GitHub Actions workflow that triggers it
4. How this fits alongside the online evaluation flywheel from Section 1

---
## 1. One dataset, synced from git, reused by every CI run

In Module 2, Section 1 we created a LangSmith dataset with a random UUID suffix (`techhub-baseline-eval-{uuid}`) — fine for an interactive notebook, but not what we want for CI: every PR would spin up its own disconnected dataset, and we'd lose the ability to compare scores across runs.

For CI, we want the opposite: **one fixed, named dataset** (`techhub-baseline-ci`) that every PR's eval run evaluates against. Since every experiment targets the same dataset, LangSmith's comparison view lets us diff correctness scores across PRs and over time — exactly like you'd expect from a regression test suite.

The tricky part: who owns the dataset's contents — git, or LangSmith? We want **git to stay the source of truth** (so a change to the eval fixture shows up as a reviewable diff in the PR, same as any other code change), so `evals/run_ci_eval.py` fully resyncs the dataset from `workshop_modules/module_2/baseline_dataset.json` on every run: delete existing examples, recreate from the JSON file. With only 12 examples this is cheap, and it guarantees there's never drift between what's committed and what CI actually evaluates against.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from evals.run_ci_eval import DATASET_NAME, DATASET_PATH, sync_dataset_from_json
from langsmith import Client

client = Client()

dataset = sync_dataset_from_json(client, DATASET_NAME, DATASET_PATH)
print(f"Dataset '{dataset.name}' synced: {dataset.url}")

Note this reuses the exact `{"question": ...} -> {"messages": [...]}` transform from Module 2, Section 1 — the dataset stores full `messages`-shaped inputs/outputs so it's directly invokable against a `MessagesState`-based agent graph like `supervisor_hitl_sql_agent`.

---
## 2. The CI eval script: `evals/run_ci_eval.py`

This is a plain Python script (not a notebook) — it needs to run non-interactively inside GitHub Actions. Structurally it's the same eval-driven-development pattern from Module 2, Section 2, just packaged for automation:

1. **Build the target agent** — `create_supervisor_hitl_agent(db_agent=create_sql_agent(), docs_agent=create_docs_agent())`, the same "improved" agent assembled in Module 2, Section 2.
2. **Sync the dataset** — as shown above.
3. **Run the same two evaluators from Module 2** — `correctness_evaluator` and `count_total_tool_calls_evaluator` — via `client.evaluate(...)`, tagged with `experiment_prefix="ci-regression"` so every CI experiment is easy to find and group in the LangSmith UI.
4. **Compute a single gating signal**: the correctness pass rate (fraction of examples scored `True`). Tool-call count is reported for visibility (an efficiency signal worth watching) but does **not** block — only correctness is treated as a hard regression.
5. **Exit non-zero if the pass rate is below `--threshold`** (default `0.8`). That's the entire mechanism GitHub Actions uses to fail the check and block merge — no special GitHub API calls needed, just a process exit code.

Let's look at the actual source:

In [ ]:
with open("../../evals/run_ci_eval.py") as f:
    print(f.read())

You can run this exact script locally at any time — which is worth doing before opening a PR, so you're not surprised by CI:

```bash
uv run python evals/run_ci_eval.py --threshold 0.8
```

---
## 3. The GitHub Actions workflow: `.github/workflows/eval-regression.yml`

This workflow is what actually invokes `run_ci_eval.py` on every relevant PR. A few things worth calling out:

- **Trigger**: `pull_request`, scoped with a `paths:` filter to `agents/**`, `tools/**`, `evaluators/**`, `deployments/**`, `evals/**`, `config.py`, and the dataset JSON itself. A PR that only touches, say, `workshop_modules/module_1/*.ipynb` won't trigger this — we only pay the LLM-call cost when agent-relevant code actually changed.
- **Vectorstore build**: the docs sub-agent needs the RAG vectorstore, so CI builds it fresh each run with the default local HuggingFace embeddings — no extra API key required, matching local dev defaults.
- **Secrets**: reuses the same `ANTHROPIC_API_KEY` / `LANGSMITH_API_KEY` repo secrets already used by `simulate_traffic.yml`, plus a dedicated `LANGSMITH_PROJECT` for CI so these traces don't mix into the main demo dashboard.
- **Gating**: no special "post a comment" step — the workflow step's exit code *is* the GitHub Actions check status. A non-zero exit from `run_ci_eval.py` shows up as a red ✗ on the PR, and (if branch protection requires this check) blocks the merge button.

In [ ]:
with open("../../.github/workflows/eval-regression.yml") as f:
    print(f.read())

---
## 4. Where this fits in the bigger picture

This CI gate and the production data flywheel from Section 1 catch different things, at different times:

| | **Offline CI gate (this section)** | **Online eval (Section 1)** |
|---|---|---|
| Runs on | Every PR, before merge | Live production traffic, continuously |
| Dataset | Small, curated, hand-labeled | Real customer traffic |
| Catches | Regressions from a *known* code change | Drift, edge cases, and failures nobody wrote a test for |
| Feedback loop | Fast (minutes), blocks merge | Slower, routes to annotation queues → golden dataset → feeds back into offline eval |

In other words: the CI gate is your safety net for changes you *know* you're making. The online eval flywheel is how you find the regressions you *didn't* know to test for — and the golden dataset it produces is exactly the kind of example that should get added to `baseline_dataset.json`, which then strengthens this CI gate going forward. The two systems reinforce each other.